# RINA AI — Entraînement LoRA + Upload HuggingFace

Ce notebook clone le repo, installe les dépendances, génère les données d'entraînement,
lance un fine-tuning LoRA, et upload le checkpoint sur HuggingFace Hub.

**Runtime requis :** T4 GPU (gratuit) ou mieux.
**Durée estimée :** ~15-30 min selon la taille des données et le modèle de base.

In [ ]:
# @title 1. Connexion HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# @title 2. Cloner le repo
!git clone https://github.com/siliconcorerina/RINA-AI.git
%cd RINA-AI

In [ ]:
# @title 3. Installer les dépendances
!pip install -r requirements.txt -q
!pip install huggingface-hub -q

In [ ]:
# @title 4. Générer un vrai dataset (ou utiliser le vôtre)
!python scripts/generate_training_data.py --count 1000 --output finetune/data/train.jsonl --eval-output finetune/data/eval.jsonl
!echo '---'
!wc -l finetune/data/train.jsonl finetune/data/eval.jsonl

In [ ]:
# @title 5. (Optionnel) Choisir le modèle de base
# Modifiez le YAML ci-dessous si vous voulez utiliser un autre modèle
!cat finetune/configs/lora_default.yaml

In [ ]:
# @title 6. Lancer l'entraînement LoRA
# Durée ~10-20 min sur T4 selon la config
!python finetune/train.py --config finetune/configs/lora_default.yaml

In [ ]:
# @title 7. Upload vers HuggingFace Hub
# Publie le checkpoint sur siliconcorerina/rina-coder-base
# (le repo HuggingFace est déjà créé avec la model card)
!python scripts/upload_to_huggingface.py \
    --model-path outputs/rina-lora \
    --repo-id siliconcorerina/rina-coder-base \
    --message "RINA Coder v1 - LoRA fine-tuned on multi-language code dataset"

In [ ]:
# @title 8. Vérification : tester que le modèle répond
from transformers import pipeline

pipe = pipeline("text-generation", model="siliconcorerina/rina-coder-base")
result = pipe("def fibonacci(n):\n    ", max_new_tokens=80)[0]["generated_text"]
print(result)

---
**Étape bonus :** Si vous voulez fusionner le LoRA dans le modèle de base :
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-base")
model = PeftModel.from_pretrained(base, "outputs/rina-lora")
merged = model.merge_and_unload()
merged.save_pretrained("outputs/rina-coder-merged")
```
Puis uploader avec `--model-path outputs/rina-coder-merged`.